In [ ]:
# ===== Track C — CELL 0 : setup =====
# কাজ: প্রতিটা figure-এর জন্য VLM দিয়ে আসল বর্ণনামূলক caption বানানো।
#
# কেন এটাই সবচেয়ে বড় সুযোগ:
#   এখন ছবির দিকটা OCR দিয়ে বোঝানো হয় — গড়ে মাত্র ৫৪ অক্ষর, প্রায় অর্থহীন।
#   দলের নিজের মাপা সংখ্যা: একই model পরিষ্কার caption-এ 0.6242, OCR-এ 0.4158।
#   অর্থাৎ শুধু লেখার মান খারাপ হওয়ায় 0.21 macro-F1 হারাচ্ছি। image+image
#   (0.5455) আর image+text (0.6317) — দুটোই এখানে আটকে আছে।
#
# ⚠️ এই caption গুলো OCR-এর বদলি নয়, বাড়তি একটা view। খারাপ caption এলেও
#    পুরনো signal নষ্ট হবে না।
#
# Accelerator: GPU T4 x2 / P100।  Internet: ON।  Input: essentials, img1280
#
# ⏱️ Resume: প্রতি ২০ batch-এ vlm_caption.parquet সেভ হয়। session কাটা পড়লে
#    Output-কে Dataset বানিয়ে পরের run-এ Input হিসেবে দাও — যেখানে থেমেছিল
#    সেখান থেকেই চলবে, আগের কাজ নষ্ট হবে না।
import os, glob, json, time, gc, sys, subprocess, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
OUT, INP = '/kaggle/working', '/kaggle/input'
T0 = time.time()
def find(n, isdir=False):
    for r in (INP, OUT):
        for h in glob.glob(f'{r}/**/{n}', recursive=True):
            if os.path.isdir(h) == isdir: return h
    return None
def tlog(*a): print(f'[{(time.time()-T0)/60:6.1f} min]', *a, flush=True)

subprocess.run(f'{sys.executable} -m pip uninstall -y -q torchao', shell=True, timeout=900)
import torch
tlog('VRAM', f'{torch.cuda.get_device_properties(0).total_memory/1e9:.0f}GB')

ESS  = os.path.dirname(find('meta_train.parquet'))
IMGD = find('img1280', isdir=True)
assert IMGD, 'img1280 নেই — আগে E0 চালাও (384px cache দিয়ে VLM চালানোর মানে নেই)'
mtr = pd.read_parquet(f'{ESS}/meta_train.parquet')
mte = pd.read_parquet(f'{ESS}/meta_test.parquet')
for d in (mtr, mte):
    for c in ['t1','t2','h1','h2']: d[c] = d[c].astype(str)

# train+test মিলিয়ে সব unique figure — একই ছবি বহু জোড়ায় আসে, তাই hash-এ dedupe
IH = sorted(set(mtr.h1[mtr.t1=='image']) | set(mtr.h2[mtr.t2=='image']) |
            set(mte.h1[mte.t1=='image']) | set(mte.h2[mte.t2=='image']))
IH = [h for h in IH if os.path.exists(f'{IMGD}/{h}.jpg')]
tlog('unique figures to caption:', len(IH))

In [ ]:
# ===== Track C — CELL 1 : caption generation (resume-able) =====
VLM_MODEL  = 'Qwen/Qwen2.5-VL-7B-Instruct'   # 3B-ও চলে; 7B নাম-ধাম (survey/instrument) ভালো ধরে
FALLBACK   = 'Qwen/Qwen2.5-VL-3B-Instruct'
VLM_PROMPT = ('Describe this astronomy figure in one sentence: what quantity is plotted, '
              'what objects, surveys or instruments are named, and what the axes show.')
# hi-res-এ প্রতি ছবিতে ~১০x বেশি visual token, তাই batch নামাতেই হবে
VLM_BATCH, VLM_MAXNEW = 2, 48
MAX_PIXELS = 1280 * 28 * 28      # ≈1.0 Mpx — VRAM-এ যতটা কুলোয়
SAVE_EVERY = 20                 # batch; ২০ x ৮ = ১৬০ ছবি পরপর সেভ
TIME_BUDGET = 10.5 * 3600       # ১২ ঘণ্টার session-এ নিরাপদ সীমা

CKPT = f'{OUT}/vlm_caption.parquet'
done = {}
_p = find('vlm_caption.parquet')      # আগের run-এর Output attach করা থাকলে সেখান থেকে
if _p:
    v = pd.read_parquet(_p)
    done = {h: c for h, c in zip(v.hash.astype(str), v.caption.fillna('').astype(str)) if c}
    tlog(f'checkpoint থেকে {len(done)} caption পাওয়া গেল')

def _save():
    pd.DataFrame({'hash': list(done), 'caption': [done[h] for h in done]}).to_parquet(CKPT)

todo = [h for h in IH if not done.get(h)]
tlog(f'বাকি {len(todo)} / {len(IH)}')

if todo:
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration
    from PIL import Image
    MODEL = VLM_MODEL
    try:
        proc = AutoProcessor.from_pretrained(MODEL, max_pixels=MAX_PIXELS)
        vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map='auto').eval()
    except Exception as e:
        print('7B লোড হলো না:', repr(e)[:160], '->', FALLBACK)
        MODEL = FALLBACK
        proc = AutoProcessor.from_pretrained(MODEL, max_pixels=MAX_PIXELS)
        vlm = Qwen2_5_VLForConditionalGeneration.from_pretrained(
            MODEL, torch_dtype=torch.float16, device_map='auto').eval()
    proc.tokenizer.padding_side = 'left'
    tlog('model ready:', MODEL)
    # 🚨 token সত্যিই বেড়েছে কিনা এখনই দেখি — ঘণ্টার পর ঘণ্টা চালিয়ে নয়
    _im = Image.open(f'{IMGD}/{IH[0]}.jpg').convert('RGB')
    _g = proc.image_processor(images=_im, return_tensors='pt').get('image_grid_thw')
    if _g is not None:
        _ms = getattr(proc.image_processor, 'merge_size', 2)
        _n = int(np.prod(_g[0].tolist()) // (_ms*_ms))
        print(f'ছবি {_im.size} -> {_n} visual token  (384px cache-এ ছিল ~112)')
        assert _n > 400, f'token মাত্র {_n} — max_pixels কার্যকর হয়নি, থামো'
    tmpl = proc.apply_chat_template(
        [{'role':'user','content':[{'type':'image'},{'type':'text','text':VLM_PROMPT}]}],
        tokenize=False, add_generation_prompt=True)

    t0 = time.time(); nb = 0
    for i in range(0, len(todo), VLM_BATCH):
        chunk = todo[i:i+VLM_BATCH]
        try:
            ims = [Image.open(f'{IMGD}/{h}.jpg').convert('RGB') for h in chunk]
            batch = proc(text=[tmpl]*len(chunk), images=ims, return_tensors='pt', padding=True)
            batch = {k:(v.to(vlm.device) if hasattr(v,'to') else v) for k,v in batch.items()}
            with torch.no_grad():
                out = vlm.generate(**batch, max_new_tokens=VLM_MAXNEW, do_sample=False)
            for h, txt in zip(chunk, proc.batch_decode(out[:, batch['input_ids'].shape[1]:],
                                                       skip_special_tokens=True)):
                done[h] = ' '.join(txt.split())
        except torch.cuda.OutOfMemoryError:
            # দু-একটা বড় ছবিতে হতে পারে — একটা একটা করে আবার চেষ্টা, তাতেও না হলে বাদ
            gc.collect(); torch.cuda.empty_cache()
            for h in chunk:
                try:
                    b = proc(text=[tmpl], images=[Image.open(f'{IMGD}/{h}.jpg').convert('RGB')],
                             return_tensors='pt')
                    b = {k:(v.to(vlm.device) if hasattr(v,'to') else v) for k,v in b.items()}
                    with torch.no_grad():
                        o = vlm.generate(**b, max_new_tokens=VLM_MAXNEW, do_sample=False)
                    done[h] = ' '.join(proc.batch_decode(o[:, b['input_ids'].shape[1]:],
                                                        skip_special_tokens=True)[0].split())
                except Exception:
                    done[h] = ''
                gc.collect(); torch.cuda.empty_cache()
        nb += 1
        if nb % SAVE_EVERY == 0:
            _save()
            el = time.time()-t0; n = i+len(chunk)
            tlog(f'  {n}/{len(todo)} | {el/60:.1f} min | eta {(len(todo)-n)*el/max(n,1)/60:.0f} min')
        if time.time() - T0 > TIME_BUDGET:
            _save()
            tlog('⏱️ সময়সীমা — এখানেই সেভ করে থামছি। Output-কে Dataset বানিয়ে আবার চালাও।')
            break
    _save()
    del vlm; gc.collect(); torch.cuda.empty_cache()

tlog('মোট caption:', len(done))

In [ ]:
# ===== Track C — CELL 2 : মান যাচাই =====
# caption গুলো আদৌ কাজের কিনা দেখার সবচেয়ে সহজ পরীক্ষা: OCR-এর তুলনায় কত লম্বা,
# আর নাম-ধাম (survey/instrument/object) কতটা এসেছে।
import re
cap = pd.read_parquet(CKPT)
cap['caption'] = cap.caption.fillna('').astype(str)
ok = cap[cap.caption.str.len() > 0]
print(f'caption আছে: {len(ok)} / {len(IH)}  ({100*len(ok)/max(len(IH),1):.1f}%)')
print(f'গড় দৈর্ঘ্য: {int(ok.caption.str.len().mean())} অক্ষর   (OCR ছিল ~54)')

p = find('ocr_384x2.parquet') or find('ocr.parquet')
if p:
    o = pd.read_parquet(p); o['ocr'] = o.ocr.fillna('').astype(str)
    print(f'তুলনায় OCR গড়: {int(o.ocr.str.len().mean())} অক্ষর')

# বড় হাতের নাম / সংখ্যা-সহ টোকেন = সম্ভাব্য survey, instrument, object নাম
NAMEY = re.compile(r'\b([A-Z]{2,}[-\w]*|[A-Z][a-z]+\s?\d+)\b')
nm = ok.caption.map(lambda s: len(set(NAMEY.findall(s))))
print(f'caption-প্রতি গড় নাম-সদৃশ টোকেন: {nm.mean():.2f}  (যত বেশি তত ভালো)')
print(f'অন্তত একটা নাম আছে এমন caption: {100*(nm>0).mean():.1f}%')

print('\n--- নমুনা ৫টা ---')
for s in ok.caption.head(5): print(' *', s[:170])

print('\n👉 Save Version → Output কে Dataset বানাও (vlm_caption.parquet)।')
print('   এই dataset পরে Track A (A2) আর plan_d_final-এ Input হিসেবে লাগবে।')
tlog('done')